# Knowledge Workflow — Multi-Agent (`src/`)

Runs the refactored multi-agent pipeline from `src/`.  
All LLM logic lives in `src/agents/`, all utilities in `src/tools/`.

| Pipeline | Command | What it does |
|---|---|---|
| **A — Extraction** | `run_extraction(collection_id)` | Phase 1 + normalization + Phase 2 |
| **B — Enrichment** | `run_enrichment(csv_path)` | MDS-Onto tagging + draw.io diagram |
| **Full** | `run_full(collection_id)` | A → B end-to-end |

Outputs saved to `outputs/<collection>/`.

## 1 · Setup

In [ ]:
# Ensure the project root is on sys.path so `src` imports resolve correctly.
import sys, os

PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Confirm we can see src/
print('Project root:', PROJECT_ROOT)
print('src/ exists :', os.path.isdir(os.path.join(PROJECT_ROOT, 'src')))

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Config — all values come from .env or environment variables.
# Edit src/config.py or set env vars to change defaults.
from src.config import (
    MODEL, LLM_BASE_URL,
    COLLECTION_ID,
    OUTPUTS_DIR,
)

print(f'Provider   : {LLM_BASE_URL}')
print(f'Model      : {MODEL}')
print(f'Collection : {COLLECTION_ID}')

## 2 · Agent & Tool Imports

Agents and tools are imported individually so they can be inspected or called in isolation.

In [ ]:
# ── Agents ───────────────────────────────────────────────────────────────────
from src.agents.extractor      import extractor_agent,   build_concept_table
from src.agents.normalizer     import normalizer_agent,  normalize_concept_list
from src.agents.schema_builder import schema_agent,      build_schema_rows
from src.agents.tagger         import tagger_agent,      tag_concepts
from src.agents.orchestrator   import run_extraction, run_enrichment, run_full

# ── Tools ────────────────────────────────────────────────────────────────────
from src.tools.zotero_client import get_collection_map, get_collection_with_text
from src.tools.csv_writer    import load_concepts, make_filename
from src.tools.drawio_builder import build_drawio_xml, add_template_pages, serialize_drawio

# ── Models ───────────────────────────────────────────────────────────────────
from src.models.concept import ConceptList, NormalizedConceptList
from src.models.schema  import SchemaValueList
from src.models.tag     import TaggedConceptBatch

print('All imports OK')

## 3 · Inspect Agents

Each PydanticAI agent exposes its result type and system prompt.

In [ ]:
agents = {
    'extractor':      extractor_agent,
    'normalizer':     normalizer_agent,
    'schema_builder': schema_agent,
    'tagger':         tagger_agent,
}

for name, agent in agents.items():
    print(f'── {name}')
    print(f'   result_type : {agent.result_type}')
    print(f'   model       : {agent.model}')
    print()

## 4 · Browse Zotero Collections

In [ ]:
coll_map = get_collection_map()
print(f'{len(coll_map)} collections found:\n')
max_len = max(len(k) for k in coll_map)
for name, key in sorted(coll_map.items()):
    print(f'  {key}  {name}')

In [ ]:
# ── Override the collection here if needed ───────────────────────────────────
# Leave as COLLECTION_ID to use the value from .env / src/config.py
TARGET_COLLECTION = COLLECTION_ID

id_to_name      = {v: k for k, v in coll_map.items()}
collection_name = id_to_name.get(TARGET_COLLECTION, TARGET_COLLECTION)
print(f'Target: "{collection_name}"  ({TARGET_COLLECTION})')

## 5 · Pipeline A — Extraction

Runs Phase 1 (per-paper concept extraction), normalization, and Phase 2 (schema population).  
Saves `concepts_*.csv` and `schema_*.csv` to `outputs/<collection>/`.

In [ ]:
extraction_result = run_extraction(TARGET_COLLECTION)

In [ ]:
import pandas as pd

concept_table = extraction_result['concept_table']
if concept_table and concept_table.rows:
    df_concepts = pd.DataFrame([
        {'paper': r.paper[:55], 'canonical': r.canonical,
         'paper_term': r.paper_term, 'relevance': r.relevance}
        for r in concept_table.rows
    ])
    print(f'Concept-paper pairs : {len(df_concepts)}')
    print(f'Unique canonicals   : {df_concepts["canonical"].nunique()}')
    display(df_concepts.head(20))

In [ ]:
normalized_concepts = extraction_result['normalized_concepts']
print(f'Normalized concepts ({len(normalized_concepts)}):')
for i, c in enumerate(normalized_concepts, 1):
    print(f'  {i:>3}. {c}')

In [ ]:
schema_rows = extraction_result['schema_rows']
print(f'Schema rows (papers): {len(schema_rows)}')
print(f'Concepts (columns)  : {len(normalized_concepts)}')

# Flatten to DataFrame for display
if schema_rows:
    columns   = ['domain', 'doi'] + normalized_concepts
    flat_rows = []
    for sr in schema_rows:
        row = {'domain': sr.domain, 'doi': sr.doi}
        row.update(sr.cells)
        flat_rows.append(row)
    df_schema = pd.DataFrame(flat_rows, columns=columns)
    preview_cols = ['domain', 'doi'] + normalized_concepts[:4]
    display(df_schema[[c for c in preview_cols if c in df_schema.columns]].head(5))

## 6 · Pipeline B — Enrichment

Tags each concept with MDS-Onto study stage + supply chain level,  
then generates a draw.io concept map with embedded library pages.

Pass a path to an existing `concepts_*.csv`, or run after Pipeline A to use the file it just saved.

In [ ]:
import glob as _glob

# Use the CSV produced by Pipeline A, or discover the latest one.
concepts_file = extraction_result.get('concepts_file', '')

if not concepts_file or not os.path.isfile(concepts_file):
    pattern = os.path.join(OUTPUTS_DIR, '**', 'concepts_*.csv')
    found   = sorted(_glob.glob(pattern, recursive=True))
    if found:
        concepts_file = found[-1]   # most recently modified
        print(f'Using latest concepts file: {concepts_file}')
    else:
        print('No concepts_*.csv found — run Pipeline A first, or set concepts_file manually.')
else:
    print(f'Using: {concepts_file}')

In [ ]:
enrichment_result = run_enrichment(concepts_file)

In [ ]:
csv_out    = enrichment_result['csv_out']
drawio_out = enrichment_result['drawio_out']

df_enriched = pd.read_csv(csv_out)
print(f'Enriched CSV : {csv_out}')
print(f'Diagram      : {drawio_out}')
print(f'Shape        : {df_enriched.shape}\n')
display(df_enriched.head(10))

### Stage distribution

In [ ]:
stage_counts = (
    df_enriched['mds:studyStage']
    .str.split(',').explode()
    .str.strip()
    .value_counts()
)
print('Concepts per MDS study stage:')
print(stage_counts.to_string())

## 7 · Full Pipeline  (A → B)

Single call that runs extraction then enrichment end-to-end.

In [ ]:
# ⚠️  This re-runs everything — API calls + PDF processing.
# Comment out and use the individual pipeline cells above for iterative work.

# full_result = run_full(TARGET_COLLECTION)
# print('Diagram:', full_result.get('drawio_out'))
print('(Full pipeline cell commented out to prevent accidental re-runs)')

## 8 · Standalone Agent Calls

Each agent can be called directly — useful for testing prompts or re-running a single step.

In [ ]:
# ── Test the extractor on a single abstract ──────────────────────────────────
test_abstract = """We demonstrate a CdSeTe/CdTe tandem solar cell with open-circuit
voltage of 886 mV and power conversion efficiency of 22.1%. The CdSeTe absorber
layer was deposited by close-space sublimation and treated with CdCl2 to improve
grain boundary passivation. Secondary ion mass spectrometry confirmed uniform
selenium incorporation throughout the absorber."""

result = extractor_agent.run_sync(
    f'Extract the top 10 concepts from this abstract.\n\nAbstract:\n{test_abstract}'
)
for c in sorted(result.data.concepts, key=lambda x: x.relevance, reverse=True):
    print(f'  {c.relevance:.2f}  {c.canonical:<30}  {c.paper_term}')

In [ ]:
# ── Test the normalizer on a small label list ────────────────────────────────
raw_labels = [
    'open circuit voltage', 'voc', 'open-circuit voltage voc',
    'device efficiency', 'power conversion efficiency', 'pce',
    'absorber material', 'absorber layer', 'absorber',
    'CdTe', 'CdSeTe', 'cdte absorber',
    'grain boundary passivation', 'passivation',
    'selenium content', 'selenium incorporation',
]

result = normalizer_agent.run_sync(
    f'Normalize and deduplicate these {len(raw_labels)} labels.\n\n'
    + '\n'.join(f'- {c}' for c in raw_labels)
)
print('Normalized:')
for c in result.data.concepts:
    print(f'  {c}')

In [ ]:
# ── Test the tagger on a few concepts ────────────────────────────────────────
test_concepts = [
    'absorber material',
    'open circuit voltage',
    'cdte thickness',
    'annealing temperature',
    'x-ray diffraction',
    'grain boundary passivation',
    'device efficiency',
]

concept_list = '\n'.join(f'- {c}' for c in test_concepts)
result = tagger_agent.run_sync(
    f'Tag each concept with mds:studyStage and mds:supplyChainLevel.\n\n{concept_list}'
)
for t in result.data.tagged_concepts:
    print(f'  {t.concept}')
    print(f'    stage : {t.mds_study_stage}')
    print(f'    level : {t.mds_supply_chain_level}')

## 9 · Output File Summary

In [ ]:
import glob as _glob
from datetime import datetime

all_outputs = sorted(
    _glob.glob(os.path.join(OUTPUTS_DIR, '**', '*'), recursive=True),
    key=os.path.getmtime,
    reverse=True,
)
files = [p for p in all_outputs if os.path.isfile(p)]

print(f'{len(files)} output files found under {OUTPUTS_DIR}/:\n')
for p in files[:20]:
    mtime = datetime.fromtimestamp(os.path.getmtime(p)).strftime('%Y-%m-%d %H:%M')
    size  = os.path.getsize(p)
    print(f'  {mtime}  {size:>9,} B  {os.path.relpath(p)}')

if len(files) > 20:
    print(f'  … and {len(files) - 20} more')